# Complaint Dispute Classification - GPU Version

This notebook is the GPU implementation used in the assignment comparison. It mirrors the CPU baseline closely while moving the dataframe and model execution path to a RAPIDS-based stack so the effect of GPU acceleration can be evaluated directly.

## GPU Runtime Context

This version targets a RAPIDS-enabled GPU environment and is written to run in Google Colab GPU sessions as well as other compatible CUDA setups.

The optional installation cell below is included because some Colab runtimes do not ship with RAPIDS preinstalled. It follows the standard RAPIDS Colab setup path documented by the project.

Reference used for this setup: RAPIDS Colab documentation  
https://docs.rapids.ai/deployment/stable/platforms/colab/

In [1]:
# Uncomment these commands in Google Colab if RAPIDS is not already installed.
# They follow the official RAPIDS Colab guidance.

# !nvidia-smi
# !git clone https://github.com/rapidsai/rapidsai-csp-utils.git
# !python rapidsai-csp-utils/colab/pip-install.py

## Scope Of The GPU Notebook

The original project included exploratory analysis, multiple model branches, text-heavy features, and tuning utilities. For this assignment, the notebook is narrowed to the parts needed for a fair CPU-versus-GPU comparison.

### Included In This Version
- Loading the training and inference complaint files.
- Cleaning key fields, standardising missing values, parsing dates, and mapping the target to `0/1`.
- Building a small structured feature set from complaint metadata and narrative availability.
- One-hot encoding the selected categorical inputs.
- Training one XGBoost classifier and evaluating it on a held-out validation split.
- Benchmarking training, validation inference, and external inference across the dataset-size sweep.

### Left Out From The Original Project
- Exploratory-analysis and managerial-insight sections.
- Alternative model families and duplicate experiments.
- TF-IDF text features and larger preprocessing branches.
- Hyperparameter search and threshold-selection logic.
- SHAP interpretation and submission-packaging cells.

This narrower version keeps the prediction task intact while making the benchmark design much easier to follow.

## How The GPU Version Differs From The CPU Baseline

The experimental design is intentionally conservative: the target variable, feature set, split logic, benchmark fractions, and model hyperparameters are kept aligned with the CPU notebook.

The main implementation changes are the use of `cudf` for dataframe operations, `cupy` for device arrays and synchronization, and GPU-backed XGBoost with `device='cuda'`. That keeps the comparison focused on execution environment rather than on a redesigned machine-learning pipeline.


## Setup And Shared Parameters

The imports below define the GPU toolchain used throughout the notebook: RAPIDS libraries for dataframe and array work, XGBoost for modeling, and the same shared constants used in the CPU baseline for features, seeds, and benchmark settings.

In [2]:
from pathlib import Path
from random import Random
from statistics import mean, stdev
from time import perf_counter

from IPython.display import display
from xgboost import XGBClassifier

try:
    import cudf
    import cupy as cp
except ImportError as exc:
    raise ImportError(
        'This notebook expects a RAPIDS-enabled environment. In Google Colab, switch to a GPU runtime and run the RAPIDS install cell near the top of the notebook.'
    ) from exc

RANDOM_STATE = 42
VALIDATION_SIZE = 0.20
SUBSET_FRACTIONS = [0.10, 0.25, 0.50, 0.75, 1.00]
BENCHMARK_REPEATS = 3
PREDICTION_THRESHOLD = 0.50
TARGET_COLUMN = 'Consumer disputed?'
ID_COLUMN = 'Complaint ID'

CATEGORICAL_COLUMNS = [
    'Product',
    'Sub-product',
    'Issue',
    'Sub-issue',
    'Company',
    'State',
    'Tags',
    'Consumer consent provided?',
    'Submitted via',
    'Company response to consumer',
    'Company public response',
    'Timely response?',
]

NUMERIC_COLUMNS = [
    'days_to_company',
    'month_received',
    'weekday_received',
    'narrative_present',
    'narrative_len',
]

MODEL_COLUMNS = CATEGORICAL_COLUMNS + NUMERIC_COLUMNS
BENCHMARK_COLUMNS = [
    'fraction',
    'train_rows',
    'validation_rows',
    'inference_rows',
    'benchmark_repeats',
    'train_seconds_mean',
    'train_seconds_std',
    'validation_seconds_mean',
    'validation_seconds_std',
    'inference_seconds_mean',
    'inference_seconds_std',
    'accuracy',
    'precision',
    'recall',
    'f1',
]
MISSING_TOKENS = {
    '', ' ', 'na', 'n/a', 'none', 'null', 'nan',
    'not available', 'not provided', 'unknown'
}
US_STATE_CODES = {
    'AL','AK','AZ','AR','CA','CO','CT','DE','DC','FL','GA','HI','ID','IL','IN','IA','KS','KY','LA','ME',
    'MD','MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ','NM','NY','NC','ND','OH','OK','OR','PA','RI',
    'SC','SD','TN','TX','UT','VT','VA','WA','WV','WI','WY','PR','VI','GU','AS','MP'
}


## Input Files And Run Configuration

The notebook reads the same training and inference CSV files as the CPU baseline through relative paths. Using identical inputs is necessary for a meaningful comparison of runtime and predictive behaviour.

In [3]:
DATA_DIR = Path('.')
TRAIN_PATH = DATA_DIR / 'complaints_training.csv'
INFERENCE_PATH = DATA_DIR / 'complaints_inference.csv'

for path in [TRAIN_PATH, INFERENCE_PATH]:
    print(f'{path}: exists={path.exists()}')

complaints_training.csv: exists=True
complaints_inference.csv: exists=True


## Cleaning And Structured Feature Engineering

These helper functions reproduce the same logical preparation steps as the CPU notebook, but with GPU-friendly dataframe operations. They standardise text fields, normalise missing-value tokens, parse complaint dates, remove duplicate records, and create the compact structured feature set used for modeling.

In [4]:
def ensure_datetime_series(series: cudf.Series) -> cudf.Series:
    if str(series.dtype).startswith('datetime64'):
        return series
    return cudf.to_datetime(series)


DATE_COLUMNS = ['Date received', 'Date sent to company']


def clean_complaints_df(df: cudf.DataFrame, is_training: bool) -> cudf.DataFrame:
    """Minimal cleaning for the assignment-specific GPU pipeline."""
    out = df.copy(deep=True)
    out.columns = [col.strip() for col in out.columns]

    string_columns = []
    for col in out.columns:
        dtype_name = str(out[col].dtype)
        if dtype_name in {'object', 'str', 'string'}:
            string_columns.append(col)

    for col in string_columns:
        out[col] = out[col].str.strip()
        token_mask = out[col].fillna('').str.lower().isin(list(MISSING_TOKENS))
        out.loc[token_mask, col] = None

    if 'Submitted via' in out.columns:
        out['Submitted via'] = out['Submitted via'].fillna('').str.strip()
        submitted_map = {
            'web': 'Web',
            'email': 'Email',
            'fax': 'Fax',
            'phone': 'Phone',
            'referral': 'Referral',
            'postal mail': 'Postal mail',
        }
        lower_submitted = out['Submitted via'].str.lower()
        for old_value, new_value in submitted_map.items():
            mask = lower_submitted == old_value
            out.loc[mask, 'Submitted via'] = new_value
        out.loc[out['Submitted via'] == '', 'Submitted via'] = None

    if 'State' in out.columns:
        state = out['State'].fillna('').str.strip().str.upper()
        valid_state = state.isin(list(US_STATE_CODES))
        state = state.where(valid_state, None)
        out['State'] = state.fillna('Unknown')

    if 'Timely response?' in out.columns:
        timely = out['Timely response?'].fillna('').str.strip().str.lower()
        out['Timely response?'] = timely.replace({'yes': 'Yes', 'no': 'No', '': None})

    if 'ZIP code' in out.columns:
        out['ZIP code'] = out['ZIP code'].astype('str').str.extract(r'(\d{5})', expand=False)

    for date_col in DATE_COLUMNS:
        if date_col in out.columns:
            out[date_col] = ensure_datetime_series(out[date_col])

    out = out.drop_duplicates()
    if ID_COLUMN in out.columns:
        out = out.drop_duplicates(subset=[ID_COLUMN], keep='first')
        out = out.sort_values(ID_COLUMN).reset_index(drop=True)

    if is_training:
        target_clean = out[TARGET_COLUMN].fillna('').astype('str').str.strip().str.lower()
        out['target'] = target_clean.map({'yes': 1, 'no': 0})
        out = out[out['target'].notnull()].copy()
        out['target'] = out['target'].astype('int8')

    return out.reset_index(drop=True)

def add_minimal_features(df: cudf.DataFrame) -> cudf.DataFrame:
    """Create the same small structured feature set as the CPU notebook."""
    out = df.copy(deep=True)

    narrative = out['Consumer complaint narrative'].fillna('')
    out['narrative_present'] = (narrative.str.len() > 0).astype('int8')
    out['narrative_len'] = narrative.str.len().fillna(0).astype('float32')

    received_dt = ensure_datetime_series(out['Date received'])
    sent_dt = ensure_datetime_series(out['Date sent to company'])
    out['days_to_company'] = (sent_dt - received_dt).dt.days.astype('float32')
    out['month_received'] = received_dt.dt.month.astype('float32')
    out['weekday_received'] = received_dt.dt.dayofweek.astype('float32')

    for col in CATEGORICAL_COLUMNS:
        out[col] = out[col].fillna('Missing').astype('str')

    for col in NUMERIC_COLUMNS:
        out[col] = out[col].fillna(-1).astype('float32')

    return out

## Data Loading And Initial Checks

The raw complaint files are loaded into GPU-backed dataframes and then passed through the cleaning and feature-generation steps. The displayed samples provide a quick check that the prepared data matches the structure expected by the CPU baseline.

In [5]:
def load_gpu_dataframe(path: Path) -> cudf.DataFrame:
    try:
        frame = cudf.read_csv(path)
        for date_col in DATE_COLUMNS:
            if date_col in frame.columns:
                frame[date_col] = ensure_datetime_series(frame[date_col])
        return frame
    except Exception:
        # Fall back when cuDF cannot reliably parse the CSV datetime columns in this environment.
        import pandas as pd

        fallback = pd.read_csv(path, parse_dates=DATE_COLUMNS)
        return cudf.from_pandas(fallback)


train_raw = load_gpu_dataframe(TRAIN_PATH)
inference_raw = load_gpu_dataframe(INFERENCE_PATH)

print('Raw training shape:', train_raw.shape)
print('Raw inference shape:', inference_raw.shape)
display(train_raw.head(2))


Raw training shape: (321430, 18)
Raw inference shape: (100, 18)


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,2015-12-17,Mortgage,Other mortgage,"Loan modification,collection,foreclosure",None,None,None,Ocwen Financial Corporation,ME,04419,"Older American, Servicemember",None,Referral,2015-12-23,Closed with explanation,Yes,Yes,1705202
1,2015-05-27,Mortgage,Other mortgage,"Loan modification,collection,foreclosure",None,None,Company chooses not to provide a public response,WELLS FARGO & COMPANY,FL,33615,Servicemember,None,Referral,2015-05-29,Closed with explanation,Yes,No,1394282


In [6]:
train_clean = add_minimal_features(clean_complaints_df(train_raw, is_training=True))
inference_clean = add_minimal_features(clean_complaints_df(inference_raw, is_training=False))

print('Clean training rows:', len(train_clean))
print('Clean inference rows:', len(inference_clean))
print('Target distribution:')
display(train_clean['target'].value_counts().sort_index())
display(train_clean[MODEL_COLUMNS + ['target']].head(3))


Clean training rows: 321430
Clean inference rows: 100
Target distribution:


target
0    257436
1     63994
Name: count, dtype: int64

,Product,Sub-product,Issue,Sub-issue,Company,State,Tags,Consumer consent provided?,Submitted via,Company response to consumer,Company public response,Timely response?,days_to_company,month_received,weekday_received,narrative_present,narrative_len,target
0,Bank account or service,(CD) Certificate of deposit,Deposits and withdrawals,Missing,AMERICAN EXPRESS COMPANY,CA,Missing,Missing,Web,Closed with explanation,Missing,Yes,0.0,1.0,2.0,0.0,0.0,0
1,Bank account or service,Checking account,Deposits and withdrawals,Missing,WELLS FARGO & COMPANY,FL,Missing,Missing,Web,Closed with explanation,Missing,Yes,0.0,1.0,2.0,0.0,0.0,1
2,Debt collection,I do not know,Disclosure verification of debt,Right to dispute notice not received,ENCORE CAPITAL GROUP INC.,AR,Missing,Missing,Web,Closed with explanation,Missing,Yes,1.0,1.0,2.0,0.0,0.0,0


## Deterministic Dataset Subsets

The same deterministic sampling approach used in the CPU notebook is repeated here for the dataset-size sweep. Matching subset logic is important because it keeps the benchmark focused on CPU-versus-GPU execution rather than on differences in sampled data.

In [7]:
def subset_row_positions(n_rows: int, fraction: float, seed: int = RANDOM_STATE) -> list[int]:
    """Pick deterministic row positions for the size sweep."""
    if not 0 < fraction <= 1:
        raise ValueError('fraction must be in the interval (0, 1].')

    target_rows = max(1, int(round(n_rows * fraction)))
    if target_rows >= n_rows:
        return list(range(n_rows))

    rng = Random(seed)
    return sorted(rng.sample(range(n_rows), target_rows))



def subset_rows(df, fraction: float, seed: int = RANDOM_STATE):
    positions = subset_row_positions(len(df), fraction, seed)
    return df.iloc[positions].reset_index(drop=True)


ACTIVE_FRACTION = 1.00
train_subset = subset_rows(train_clean, ACTIVE_FRACTION)
inference_subset = subset_rows(inference_clean, ACTIVE_FRACTION)

print('Active subset fraction:', ACTIVE_FRACTION)
print('Training rows after subsetting:', len(train_subset))
print('Inference rows after subsetting:', len(inference_subset))


Active subset fraction: 1.0
Training rows after subsetting: 321430
Inference rows after subsetting: 100


## Train-Validation Split And Encoded Features

The cleaned subset is divided into training and validation partitions with the same stratified procedure used on CPU. The predictors are then one-hot encoded on the GPU and aligned across training, validation, and external inference sets so the model sees a consistent feature layout.

In [8]:
def encode_features_gpu(df: cudf.DataFrame) -> cudf.DataFrame:
    encoded = cudf.get_dummies(df[MODEL_COLUMNS].copy(), columns=CATEGORICAL_COLUMNS, dtype='int8')
    encoded = encoded[sorted(encoded.columns)]
    return encoded



def align_to_reference_columns(encoded_df: cudf.DataFrame, reference_columns) -> cudf.DataFrame:
    aligned = encoded_df.copy(deep=True)
    for col in reference_columns:
        if col not in aligned.columns:
            aligned[col] = 0
    extra_cols = [col for col in aligned.columns if col not in reference_columns]
    if extra_cols:
        aligned = aligned.drop(columns=extra_cols)
    return aligned[reference_columns]



def stratified_train_validation_indices(labels, validation_size: float, seed: int = RANDOM_STATE):
    label_positions = {}
    for idx, value in enumerate(labels):
        label_positions.setdefault(int(value), []).append(idx)

    rng = Random(seed)
    train_idx = []
    val_idx = []

    for label in sorted(label_positions):
        positions = label_positions[label][:]
        rng.shuffle(positions)

        if len(positions) <= 1:
            val_count = 0
        else:
            val_count = int(round(len(positions) * validation_size))
            val_count = max(1, min(len(positions) - 1, val_count))

        val_idx.extend(sorted(positions[:val_count]))
        train_idx.extend(sorted(positions[val_count:]))

    return sorted(train_idx), sorted(val_idx)



def prepare_gpu_datasets(train_df: cudf.DataFrame, inference_df: cudf.DataFrame):
    target = [int(value) for value in train_df['target'].to_cupy().tolist()]
    train_idx, val_idx = stratified_train_validation_indices(target, VALIDATION_SIZE, seed=RANDOM_STATE)

    train_part = train_df.iloc[train_idx].reset_index(drop=True)
    val_part = train_df.iloc[val_idx].reset_index(drop=True)

    X_train = encode_features_gpu(train_part)
    X_val = encode_features_gpu(val_part)
    X_inference = encode_features_gpu(inference_df)

    reference_columns = list(X_train.columns)
    X_val = align_to_reference_columns(X_val, reference_columns)
    X_inference = align_to_reference_columns(X_inference, reference_columns)

    y_train = train_part['target'].to_cupy()
    y_val = val_part['target'].to_cupy()

    return {
        'train_part': train_part,
        'val_part': val_part,
        'X_train': X_train,
        'X_val': X_val,
        'X_inference': X_inference,
        'y_train': y_train,
        'y_val': y_val,
        'feature_columns': reference_columns,
    }



datasets = prepare_gpu_datasets(train_subset, inference_subset)
print('Training matrix shape:', datasets['X_train'].shape)
print('Validation matrix shape:', datasets['X_val'].shape)
print('Inference matrix shape:', datasets['X_inference'].shape)
print('Encoded feature count:', len(datasets['feature_columns']))


Training matrix shape: (257144, 3375)
Validation matrix shape: (64286, 3375)
Inference matrix shape: (100, 3375)
Encoded feature count: 3375


## Model Fitting, Validation, And Inference

This section trains the GPU XGBoost model, evaluates it on the held-out validation set, and generates predictions for the separate inference file. GPU synchronization is included around timed operations so the recorded wall-clock measurements reflect actual device work rather than asynchronous kernel launch alone.

In [9]:
def ensure_cupy_1d(values):
    if isinstance(values, cp.ndarray):
        return values
    if hasattr(values, 'to_cupy'):
        return values.to_cupy()
    return cp.asarray(values)



def build_gpu_model(y_train) -> XGBClassifier:
    target = ensure_cupy_1d(y_train)
    negatives = int(cp.sum(target == 0).item())
    positives = int(cp.sum(target == 1).item())
    scale_pos_weight = negatives / max(positives, 1)

    return XGBClassifier(
        n_estimators=600,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.80,
        colsample_bytree=0.80,
        objective='binary:logistic',
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        tree_method='hist',
        device='cuda',
        scale_pos_weight=scale_pos_weight,
    )



def evaluate_binary(y_true, positive_scores, threshold: float = PREDICTION_THRESHOLD) -> dict:
    y_true_device = ensure_cupy_1d(y_true)
    score_device = ensure_cupy_1d(positive_scores)
    predictions = (score_device >= threshold).astype(cp.int8)

    tp = int(cp.sum((predictions == 1) & (y_true_device == 1)).item())
    tn = int(cp.sum((predictions == 0) & (y_true_device == 0)).item())
    fp = int(cp.sum((predictions == 1) & (y_true_device == 0)).item())
    fn = int(cp.sum((predictions == 0) & (y_true_device == 1)).item())

    total = tp + tn + fp + fn
    accuracy = (tp + tn) / total if total else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0

    return {
        'threshold': threshold,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'predictions': predictions,
    }



def gpu_synchronize() -> None:
    cp.cuda.runtime.deviceSynchronize()


gpu_model = build_gpu_model(datasets['y_train'])
gpu_model.fit(datasets['X_train'], datasets['y_train'])
gpu_synchronize()

val_scores = ensure_cupy_1d(gpu_model.predict_proba(datasets['X_val'])[:, 1])
val_metrics = evaluate_binary(datasets['y_val'], val_scores, threshold=PREDICTION_THRESHOLD)
inference_scores = ensure_cupy_1d(gpu_model.predict_proba(datasets['X_inference'])[:, 1])
inference_predictions = (inference_scores >= PREDICTION_THRESHOLD).astype(cp.int8)

metrics_table = cudf.DataFrame([
    {
        'accuracy': val_metrics['accuracy'],
        'precision': val_metrics['precision'],
        'recall': val_metrics['recall'],
        'f1': val_metrics['f1'],
        'validation_rows': len(datasets['y_val']),
        'inference_rows': len(inference_predictions),
    }
])

prediction_sample = inference_subset[[ID_COLUMN]].reset_index(drop=True).copy(deep=True)
prediction_sample['gpu_probability'] = cudf.Series(inference_scores)
prediction_sample['gpu_prediction'] = cudf.Series(inference_predictions)

display(metrics_table)
display(prediction_sample.head(10))


,accuracy,precision,recall,f1,validation_rows,inference_rows
0,0.564353,0.267342,0.682631,0.384213,64286,100


,Complaint ID,gpu_probability,gpu_prediction
0,2271627,0.622417,1
1,2274992,0.616449,1
2,2289332,0.561532,1
3,2289336,0.322760,0
4,2312355,0.436189,0
5,2331272,0.469352,0
6,2340546,0.594721,1
7,2356986,0.655442,1
8,2372244,0.700443,1
9,2379919,0.541369,1


## GPU Output Bundle

The notebook stores a compact summary of the GPU run in the same structure used by the CPU notebook.

That bundle records row counts, encoded columns, prediction lengths, validation metrics, and probability vectors so the two implementations can be checked for structural and predictive equivalence.


In [10]:
def build_gpu_artifact_bundle() -> dict:
    return {
        'fraction': ACTIVE_FRACTION,
        'train_rows_after_cleaning': int(len(train_subset)),
        'inference_rows_after_cleaning': int(len(inference_subset)),
        'feature_columns': list(datasets['feature_columns']),
        'validation_prediction_length': int(len(val_metrics['predictions'])),
        'inference_prediction_length': int(len(inference_predictions)),
        'metrics': {
            'accuracy': float(val_metrics['accuracy']),
            'precision': float(val_metrics['precision']),
            'recall': float(val_metrics['recall']),
            'f1': float(val_metrics['f1']),
        },
        'validation_probabilities': val_scores.tolist(),
        'inference_probabilities': inference_scores.tolist(),
    }


gpu_artifact_bundle = build_gpu_artifact_bundle()
gpu_artifact_overview = cudf.DataFrame([
    {
        'fraction': gpu_artifact_bundle['fraction'],
        'train_rows_after_cleaning': gpu_artifact_bundle['train_rows_after_cleaning'],
        'inference_rows_after_cleaning': gpu_artifact_bundle['inference_rows_after_cleaning'],
        'encoded_feature_count': len(gpu_artifact_bundle['feature_columns']),
        'validation_prediction_length': gpu_artifact_bundle['validation_prediction_length'],
        'inference_prediction_length': gpu_artifact_bundle['inference_prediction_length'],
        'accuracy': gpu_artifact_bundle['metrics']['accuracy'],
        'precision': gpu_artifact_bundle['metrics']['precision'],
        'recall': gpu_artifact_bundle['metrics']['recall'],
        'f1': gpu_artifact_bundle['metrics']['f1'],
    }
])

display(gpu_artifact_overview)


,fraction,train_rows_after_cleaning,inference_rows_after_cleaning,encoded_feature_count,validation_prediction_length,inference_prediction_length,accuracy,precision,recall,f1
0,1.0,321430,100,3375,64286,100,0.564353,0.267342,0.682631,0.384213


## Benchmark Sweep

The final stage repeats the GPU pipeline at 10%, 25%, 50%, 75%, and 100% of the cleaned data.

Training time, validation inference time, and external inference time are recorded separately, with three repeats per size, so the later analysis can show how GPU benefits change with scale and workload type.


In [11]:
def time_block(func, sync_fn=None):
    if sync_fn is not None:
        sync_fn()
    start = perf_counter()
    result = func()
    if sync_fn is not None:
        sync_fn()
    elapsed = perf_counter() - start
    return result, elapsed



def summarize_timings(values) -> tuple[float, float]:
    if len(values) == 1:
        return float(values[0]), 0.0
    return float(mean(values)), float(stdev(values))



def run_gpu_pipeline_for_fraction(fraction: float, repeats: int = BENCHMARK_REPEATS) -> dict:
    sampled_train = subset_rows(train_clean, fraction)
    sampled_inference = subset_rows(inference_clean, fraction)

    train_timings = []
    validation_timings = []
    inference_timings = []
    last_metrics = None
    last_datasets = None

    for _ in range(repeats):
        sampled_datasets = prepare_gpu_datasets(sampled_train, sampled_inference)
        model = build_gpu_model(sampled_datasets['y_train'])

        _, train_seconds = time_block(
            lambda: model.fit(sampled_datasets['X_train'], sampled_datasets['y_train']),
            sync_fn=gpu_synchronize,
        )
        validation_scores_local, validation_seconds = time_block(
            lambda: ensure_cupy_1d(model.predict_proba(sampled_datasets['X_val'])[:, 1]),
            sync_fn=gpu_synchronize,
        )
        _, inference_seconds = time_block(
            lambda: ensure_cupy_1d(model.predict_proba(sampled_datasets['X_inference'])[:, 1]),
            sync_fn=gpu_synchronize,
        )

        train_timings.append(train_seconds)
        validation_timings.append(validation_seconds)
        inference_timings.append(inference_seconds)
        last_metrics = evaluate_binary(sampled_datasets['y_val'], validation_scores_local)
        last_datasets = sampled_datasets

    train_mean, train_std = summarize_timings(train_timings)
    validation_mean, validation_std = summarize_timings(validation_timings)
    inference_mean, inference_std = summarize_timings(inference_timings)

    return {
        'fraction': fraction,
        'train_rows': len(sampled_train),
        'validation_rows': len(last_datasets['y_val']),
        'inference_rows': len(sampled_inference),
        'benchmark_repeats': repeats,
        'train_seconds_mean': train_mean,
        'train_seconds_std': train_std,
        'validation_seconds_mean': validation_mean,
        'validation_seconds_std': validation_std,
        'inference_seconds_mean': inference_mean,
        'inference_seconds_std': inference_std,
        'accuracy': last_metrics['accuracy'],
        'precision': last_metrics['precision'],
        'recall': last_metrics['recall'],
        'f1': last_metrics['f1'],
    }



def run_gpu_benchmark() -> cudf.DataFrame:
    results = [run_gpu_pipeline_for_fraction(fraction) for fraction in SUBSET_FRACTIONS]
    return cudf.DataFrame(results)[BENCHMARK_COLUMNS]


gpu_benchmark_results = run_gpu_benchmark()
display(gpu_benchmark_results.round(4))


,fraction,train_rows,validation_rows,inference_rows,benchmark_repeats,train_seconds_mean,train_seconds_std,validation_seconds_mean,validation_seconds_std,inference_seconds_mean,inference_seconds_std,accuracy,precision,recall,f1
0,0.10,32143,6429,10,3,3.2009,0.3171,0.1717,0.0197,0.1908,0.0973,0.5970,0.2630,0.5692,0.3598
1,0.25,80358,16072,25,3,4.7043,0.0907,0.2528,0.0946,0.1783,0.0126,0.5729,0.2655,0.6409,0.3754
2,0.50,160715,32143,50,3,7.6148,0.1740,0.3286,0.0947,0.2873,0.0918,0.5698,0.2678,0.6635,0.3816
3,0.75,241072,48214,75,3,10.8998,0.0262,0.2771,0.0115,0.4284,0.0157,0.5630,0.2686,0.6879,0.3863
4,1.00,321430,64286,100,3,17.3749,3.1617,0.7288,0.0603,0.4499,0.1565,0.5645,0.2675,0.6831,0.3845
